In [1]:
import json
from pathlib import Path

import pandas as pd
import plotly.express as px
from transformers import PreTrainedTokenizer

from pivotal_tokens.constants import get_data_dir, get_artifacts_dir
from pivotal_tokens.hf.loading import  load_tokenizer
from pivotal_tokens.repo import SampleRepo, DictRepo


# EXP_DIR = get_data_dir() / 'experiments' / 'exp2.1-sps-tokens' / 'exp2.1.2-qwen3-1.7b-sps-tokens-small-prob-threshold'
# EXP_DIR = get_data_dir() / 'experiments' / 'exp2.1-sps-tokens' / 'exp2.1.3-qwen3-1.7b-sps-tokens-small-prob-threshold'
EXP_DIR = get_data_dir() / 'experiments' / 'exp2.1-sps-tokens' / 'exp2.1.4-qwen3-1.7b-sps-tokens-small-prob-threshold'
REPO_DIR = EXP_DIR / 'data' / 'repo'
PIVOTAL_TOKENS_FILE = EXP_DIR / 'data' / 'pivotal_tokens.csv'

TRACES_FILE = get_artifacts_dir() / 'exp1.1.1-qwen3-1.7b-traces.csv'

QWEN3_1_7B_MODEL_ID = 'Qwen/Qwen3-1.7B'

In [2]:
tokenizer = load_tokenizer(QWEN3_1_7B_MODEL_ID)

In [3]:
base_repo = DictRepo(dirpath=REPO_DIR)

In [4]:
traces_df = pd.read_csv(TRACES_FILE)
traces_df[:2]

,id,query,ground_truth,trace
0,5a8b57f25542995d1e6f1371,Were Scott Derrickson and Ed Wood of the same ...,yes,"<think>\nOkay, let's see. The question is whet..."
1,5a8db19d5542994ba4e3dd00,Are Local H and For Against both from the Unit...,yes,"<think>\nOkay, let's see. The question is aski..."


In [5]:
tokens_df = pd.read_csv(PIVOTAL_TOKENS_FILE)
tokens_df['token_ids'] = tokens_df['token_ids'].apply(json.loads)
tokens_df[:2]

,sample_id,span_id,token_ids,span_text,prob_before,prob_after,prob_delta,is_pivotal,pivotal_context,metadata
0,5a8db19d5542994ba4e3dd00,fc805f90-e736-11f0-801c-08bfb8a7c92d,[264],a,0.46875,0.40625,-0.06250,False,<|im_start|>system\nAnswer the following quest...,"{'id': '5a8db19d5542994ba4e3dd00', 'question':..."
1,5a8db19d5542994ba4e3dd00,feb657ba-e736-11f0-801c-08bfb8a7c92d,[4948],political,0.40625,0.93750,0.53125,True,<|im_start|>system\nAnswer the following quest...,"{'id': '5a8db19d5542994ba4e3dd00', 'question':..."


In [6]:
df = pd.merge(traces_df, tokens_df, left_on='id', right_on='sample_id', how='inner')
df[:2]

,id,query,ground_truth,trace,sample_id,span_id,token_ids,span_text,prob_before,prob_after,prob_delta,is_pivotal,pivotal_context,metadata
0,5a8db19d5542994ba4e3dd00,Are Local H and For Against both from the Unit...,yes,"<think>\nOkay, let's see. The question is aski...",5a8db19d5542994ba4e3dd00,fc805f90-e736-11f0-801c-08bfb8a7c92d,[264],a,0.46875,0.40625,-0.06250,False,<|im_start|>system\nAnswer the following quest...,"{'id': '5a8db19d5542994ba4e3dd00', 'question':..."
1,5a8db19d5542994ba4e3dd00,Are Local H and For Against both from the Unit...,yes,"<think>\nOkay, let's see. The question is aski...",5a8db19d5542994ba4e3dd00,feb657ba-e736-11f0-801c-08bfb8a7c92d,[4948],political,0.40625,0.93750,0.53125,True,<|im_start|>system\nAnswer the following quest...,"{'id': '5a8db19d5542994ba4e3dd00', 'question':..."


In [7]:
def get_thinking_trace_from_completion(completion: str) -> str:
    prefix_split_str = '<|im_start|>assistant\n'
    completion_with_suffix = completion.split(prefix_split_str)[-1].strip()

    suffix_split_str = '<|im_end|>'
    completion = completion_with_suffix.split(suffix_split_str)[0].strip()
    return completion


def get_token_length(text: str, tokenizer: PreTrainedTokenizer) -> int:
    token_ids = tokenizer.encode(text, add_special_tokens=False)
    return len(token_ids)

In [8]:
df['partial_trace'] = df['pivotal_context'].apply(get_thinking_trace_from_completion)
df['partial_trace_token_length'] = df['partial_trace'].apply(lambda x: get_token_length(x, tokenizer))
df['trace_token_length'] = df['trace'].apply(lambda x: get_token_length(x, tokenizer))
df['pivotal_token_relative_position'] = (df['partial_trace_token_length'] + 1) / df['trace_token_length']

assert (df['partial_trace_token_length'] < df['trace_token_length']).mean() == 1.0

In [9]:
def get_init_success_prob(sample_id: str, base_repo: DictRepo) -> float | None:
    repo = SampleRepo(base_repo=base_repo, sample_id=sample_id)
    subdivisions = repo.list(path="subdivisions")

    init_prob = None
    for subdiv in subdivisions:
        data = repo.load(path="subdivisions", key=subdiv)
        if len(data['prefix']) == 0 and \
                data['full_seq'].startswith('<think>') and \
                data['full_seq'].endswith('</think>'):
            init_prob = data['prob_before']
            break
    
    return init_prob

In [10]:
df['prob_init'] = df['sample_id'].apply(lambda x: get_init_success_prob(x, base_repo))

In [11]:
df['prob_before_normalized'] = df['prob_before'] / df['prob_init']
df['prob_after_normalized'] = df['prob_after'] / df['prob_init']
df['prob_delta_normalized'] = df['prob_delta'] / df['prob_init']

In [12]:
df['is_positive'] = df['prob_delta'] >= 0

In [13]:
df = df[df['is_pivotal'] == True].copy()

* * *

In [14]:
print(f"Number of pivotal tokens: {len(df)}")

df['is_positive'].value_counts()

Number of pivotal tokens: 605


is_positive
True     414
False    191
Name: count, dtype: int64

In [15]:
# df = df[df['is_positive'] == True].copy()

In [16]:
counts = df.groupby('sample_id', as_index=False).agg({'span_id': 'nunique'})
px.histogram(counts, x='span_id', nbins=20, title='Number of Positive Pivotal Tokens per Sample').show()

In [17]:
# Check relation between pivotal token position and trace length
fig = px.scatter(df,
                x='trace_token_length',
                y='partial_trace_token_length',
                hover_data=['sample_id', 'prob_before', 'prob_after', 'prob_init', 'span_text'],
                title='Partial Trace Token Length vs Full Trace Token Length')
fig.update_layout(width=700)
fig.show()

In [18]:
# Check if longer traces correlate with lower initial success probability
fig = px.scatter(df,
                 x='trace_token_length' ,
                 y='prob_init',
                 hover_data=['sample_id', 'prob_before', 'prob_after'],
                 title='Initial Success Probability vs Full Trace Token Length'
                 )
fig.update_layout(width=700)
fig.show()

In [19]:
# Check relation between pivotal token position and its impact on success probability
fig = px.scatter(df,
                 x='partial_trace_token_length' ,
                 y='prob_delta',
                 marginal_x='histogram',
                 hover_data=['sample_id', 'prob_before', 'prob_after', 'prob_init', 'span_text'],
                 title='Pivotal Token Position vs Impact on Success Probability'
                 )
fig.update_layout(width=700)
fig.show()


fig = px.scatter(df,
                 x='pivotal_token_relative_position' ,
                 y='prob_delta',
                 marginal_x='histogram',
                 hover_data=['sample_id', 'prob_before', 'prob_after', 'prob_init', 'span_text'],
                 title='Pivotal Token Position vs Impact on Success Probability'
                 )
fig.update_layout(width=700)
fig.show()

In [20]:
# Check relation between pivotal token position and its normalized impact on success probability
from turtle import width


fig = px.scatter(df,
                 x='partial_trace_token_length' ,
                 y='prob_delta_normalized',
                 marginal_x='histogram',
                 hover_data=['sample_id', 'prob_before', 'prob_after', 'prob_init', 'span_text'],
                 title='Pivotal Token Position vs Normalized Impact on Success Probability'
                 )
fig.update_layout(width=700)
fig.show()


fig = px.scatter(df,
                 x='pivotal_token_relative_position' ,
                 y='prob_delta_normalized',
                 marginal_x='histogram',
                 hover_data=['sample_id', 'prob_before', 'prob_after', 'prob_init', 'span_text'],
                 title='Pivotal Token Position vs Normalized Impact on Success Probability'
                 )
fig.update_layout(width=700)
fig.show()

In [21]:
# Check relation between pivotal token position and trace length
fig = px.scatter(df,
                x='prob_delta',
                y='prob_init',
                marginal_y='histogram',
                hover_data=['sample_id', 'prob_before', 'prob_after', 'prob_init', 'span_text'],
                title='Initial Success Probability vs Probability Delta')
fig.update_layout(width=700)
fig.show()

In [22]:
fig = px.histogram(df, x='prob_delta', nbins=50, title='Histogram of Probability Delta')
fig.show()

In [23]:
fig = px.histogram(df, x='prob_delta_normalized', nbins=50, title='Histogram of Normalized Probability Delta')
fig.show()

In [24]:
# df[df['partial_trace_token_length'] == 1]

In [25]:
# TODO:
# 1. Make a list of what I've checked before for the pivotal tokens analysis
# 2. Reproduce all the plots and tables, analyze results, make conclusions. Compare to previous conclusions.
# 3. Prepare random tokens to replace pivotal tokens.
# 4. Prepare script to run the experiments with replaced tokens.
# 5. Run the experiments. 